In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Görüntüleri kopyala
if not os.path.exists('/content/train_images'):
    print("Kopyalanıyor...")
    shutil.copytree(
        '/content/drive/MyDrive/KemikAI/data/raw/RSNA_train/images',
        '/content/train_images'
    )
    print("✅ Kopyalandı!")
else:
    print("✅ Zaten var!")

# CSV yükle
df = pd.read_csv('/content/drive/MyDrive/KemikAI/data/raw/RSNA_Annotations/RSNA_Annotations/BONEAGE/boneage_train.csv')
df['gender'] = df['Male'].apply(lambda x: 1 if x else 0)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Mounted at /content/drive
Kopyalanıyor...
✅ Kopyalandı!
Train: 10088, Val: 2523


In [2]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. KONFİGÜRASYON VE PARAMETRELER
# ==========================================
IMAGE_DIR = '/content/train_images' # Resimlerin ana klasörü
MODEL_SAVE_DIR = '/content/drive/MyDrive/KemikAI/models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# LÜTFEN DİKKAT: Mevcut Colab ortamınızdaki "train_df" DataFrame'ini kullanacağız.
# Eğitime başlamadan önce notebook'ta train_df değişkeninin yüklü olduğundan emin olun.
df = train_df.copy() # type: ignore

# ==========================================
# 2. BİYOLOJİK SENTETİK KLİNİK VERİ ÜRETİMİ
# ==========================================
def generate_synthetic_clinical_data(df):
    print("Sentetik klinik veriler biyolojik gelişim eğrilerine göre üretiliyor...")
    df = df.copy()

    # Boneage (Ay) bilgisini Yıl'a çevir
    age_y = df['Boneage'] / 12.0

    # 1. Mevcut Boy (Ortalama 70cm doğum boyu + yıllık 6cm uzama baz alınarak)
    base_height = 70 + (age_y * 6)
    df['height_cm'] = np.random.normal(base_height, 5).clip(50, 200)

    # 2. Mevcut Kilo (Ortalama 10kg + yıllık 2.5kg artış baz alınarak)
    base_weight = 10 + (age_y * 2.5)
    df['weight_kg'] = np.random.normal(base_weight, 3).clip(3, 120)

    # 3. Anne - Baba Boyu (Genetik potansiyel olarak rastgele normal dağılım)
    df['mother_height'] = np.random.normal(162, 6, size=len(df)).clip(145, 185)
    df['father_height'] = np.random.normal(175, 7, size=len(df)).clip(160, 195)

    # 4. Kategorik Klinik Değerler
    df['vit_d_deficiency'] = np.random.choice([0, 1], p=[0.8, 0.2], size=len(df))
    df['calcium_level'] = np.random.choice([0, 1, 2], p=[0.1, 0.8, 0.1], size=len(df))
    df['fracture_history'] = np.random.choice([0, 1], p=[0.85, 0.15], size=len(df))

    # 5. Mevcut Cinsiyet bilgisini garanti altına alma
    df['gender'] = df['gender'] if 'gender' in df.columns else df['Male'].astype(int)

    return df

df = generate_synthetic_clinical_data(df)

# ==========================================
# 3. DOSYA YOLLARI VE ÖN İŞLEME (PREPROCESSING)
# ==========================================
def get_existing_path(x):
    filename = f"{x}.png" if not str(x).endswith('.png') else str(x)
    paths = [
        os.path.join(IMAGE_DIR, 'RSNA_train', 'images', filename),
        os.path.join(IMAGE_DIR, 'RSNA_val', 'images', filename),
        os.path.join(IMAGE_DIR, 'RSNA_train', filename),
        os.path.join(IMAGE_DIR, 'RSNA_val', filename),
        os.path.join(IMAGE_DIR, filename)
    ]
    for p in paths:
        if os.path.exists(p): return p
    return paths[0]

df['file_path'] = df['ID'].apply(get_existing_path)
exists_mask = df['file_path'].apply(os.path.exists)
missing_count = (~exists_mask).sum()
if missing_count > 0:
    print(f"UYARI: {missing_count} resim diskte bulunamadı.")
df = df[exists_mask]

# Train / Val Split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Tabular (Klinik) Özellikleri Seçme
continuous_cols = ['height_cm', 'weight_kg', 'mother_height', 'father_height']
categorical_cols = ['gender', 'vit_d_deficiency', 'calcium_level', 'fracture_history']
tabular_cols = continuous_cols + categorical_cols

print(f"Eğitimde kullanılacak Tabular (Klinik) kolonlar: {tabular_cols}")

# ÖNEMLİ: Sayısal değişkenleri Z-Score ile Normalize et (Aksi halde gradient patlar)
scaler = StandardScaler()
# Sadece Train verisi üzerinden scaler fit edilir (Veri sızıntısını önlemek için)
train_df[continuous_cols] = scaler.fit_transform(train_df[continuous_cols])
val_df[continuous_cols] = scaler.transform(val_df[continuous_cols])

# ==========================================
# 4. TF.DATA.DATASET OLUŞTURMA
# ==========================================
def create_multimodal_dataset(df, is_training=True):
    file_paths = df['file_path'].values
    tabular_data = df[tabular_cols].values.astype(np.float32)
    labels = df['Boneage'].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((file_paths, tabular_data, labels))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(df))

    def process_path(file_path, tabular, label):
        # 1. Resmi oku
        img = tf.io.read_file(file_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        # 2. Resim ve tabular veriyi Tuple olarak döndür
        return ({"image_input": img, "tabular_input": tabular}, label)

    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = create_multimodal_dataset(train_df, is_training=True)
val_dataset = create_multimodal_dataset(val_df, is_training=False)

# ==========================================
# 5. MULTIMODAL MİMARİ
# ==========================================
# Dal 1: Görüntü (Image Branch)
image_input = layers.Input(shape=(224, 224, 3), name='image_input')
base_model = applications.EfficientNetV2S(include_top=False, input_tensor=image_input, weights='imagenet')
base_model.trainable = False  # İlk aşama için dondur

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(256, activation='relu')(x)
image_branch_out = layers.Dropout(0.3)(x)

# Dal 2: Klinik (Tabular Branch)
tabular_input = layers.Input(shape=(len(tabular_cols),), name='tabular_input')
y = layers.Dense(64, activation='relu')(tabular_input)
y = layers.Dropout(0.2)(y)
tabular_branch_out = layers.Dense(32, activation='relu')(y)

# Dal 3: Birleştirme (Fusion)
combined = layers.Concatenate()([image_branch_out, tabular_branch_out])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
final_output = layers.Dense(1, activation='linear', name='boneage_output')(z)

model = models.Model(inputs=[image_input, tabular_input], outputs=final_output)
print("\nMultimodal Mimari Özeti:")
model.summary()

# ==========================================
# 6. EĞİTİM - AŞAMA 1 (Warm-up)
# ==========================================
print("\n--- AŞAMA 1: Sadece Yeni Katmanların ve Klinik Dalın Eğitimi ---")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=tf.keras.losses.Huber(),
              metrics=['mae'])

model.fit(train_dataset, validation_data=val_dataset, epochs=5)

# ==========================================
# 7. EĞİTİM - AŞAMA 2 (Fine-tuning)
# ==========================================
print("\n--- AŞAMA 2: Tüm Modelin İnce Ayarı (Fine-tuning) ---")
base_model.trainable = True

# Loss Spike'ı önlemek için BatchNormalization katmanlarını tekrar dondur
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss=tf.keras.losses.Huber(),
              metrics=['mae'])

callbacks = [
    ModelCheckpoint(os.path.join(MODEL_SAVE_DIR, 'kemikai_multimodal_best.keras'),
                    monitor='val_mae', save_best_only=True, mode='min', verbose=1),
    EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True, mode='min', verbose=1),
    ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

model.fit(train_dataset, validation_data=val_dataset, epochs=24, callbacks=callbacks)

print("\n🎉 Eğitim Başarıyla Tamamlandı! Model Drive'a 'kemikai_multimodal_best.keras' olarak kaydedildi.")


Sentetik klinik veriler biyolojik gelişim eğrilerine göre üretiliyor...
UYARI: 585 resim diskte bulunamadı.
Eğitimde kullanılacak Tabular (Klinik) kolonlar: ['height_cm', 'weight_kg', 'mother_height', 'father_height', 'gender', 'vit_d_deficiency', 'calcium_level', 'fracture_history']
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

Multimodal Mimari Özeti:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 112, 112,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 112, 112,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 112, 112,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 112, 112,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 56, 56,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 56, 56,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 56, 56,    │          0 │ block2a_expand_b

 Total params: 20,699,073 (78.96 MB)

 Trainable params: 367,713 (1.40 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


--- AŞAMA 1: Sadece Yeni Katmanların ve Klinik Dalın Eğitimi ---
Epoch 1/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 353s 1s/step - loss: 28.3978 - mae: 28.8936 - val_loss: 10.9553 - val_mae: 11.4457
Epoch 2/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - loss: 14.2756 - mae: 14.7694 - val_loss: 8.1216 - val_mae: 8.6059
Epoch 3/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 268s 1s/step - loss: 13.3876 - mae: 13.8788 - val_loss: 7.4881 - val_mae: 7.9727
Epoch 4/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 233s 980ms/step - loss: 12.6800 - mae: 13.1719 - val_loss: 7.5312 - val_mae: 8.0216
Epoch 5/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 233s 977ms/step - loss: 12.3184 - mae: 12.8104 - val_loss: 6.4413 - val_mae: 6.9262

--- AŞAMA 2: Tüm Modelin İnce Ayarı (Fine-tuning) ---
Epoch 1/24
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 12.5545 - mae: 13.0456   
Epoch 1: val_mae improved from None to 6.37641, saving model to /content/drive/MyDrive/KemikAI/models/kemikai_multimodal_best.keras

Epoch 1: finished saving model to /content/drive/MyDr